In [41]:
#Step 7 to convert cell-based model to subarea-based model

In [42]:
import pandas as pd
import os
import numpy as np
import sqlite3

In [43]:
#subarea data file name to average column name dictionary
file_column_dic = {
    'usle_p.txt':'USLE_P',
    'moist_in.txt':'MoistureInitial',
    'flow_acc.txt':'FlowAccumulationAverage',
    'wetland.txt':'WetlandFraction',
    'IUH002Yr_t0s.txt':'TravelTimeAverage2',
    'IUH010Yr_t0s.txt':'TravelTimeAverage10',
    'IUH100Yr_t0s.txt':'TravelTimeAverage100',
    'IUH002Yr_delta_t.txt':'TravelTimeStd2',
    'IUH010Yr_delta_t.txt':'TravelTimeStd10',
    'IUH100Yr_delta_t.txt':'TravelTimeStd100'
}

def getColumnFromFile(filename):
    if filename in file_column_dic:
        return file_column_dic[filename]
    return "unknown"

In [44]:
#for STC
#cell_subarea_csv = r"D:\git\imWEBs\IMWEBsInterface\resources\DemoDatasets\STC2021\SubArea\database\CellSubArea.csv"
#subarea_data_path = r"D:\imWEBs\STC\model\STC-Catchabasin-Reservoir-Separated\Model01-SubArea\SubareaData"
#subarea_csv_path = r"D:\git\imWEBs\IMWEBsInterface\resources\DemoDatasets\STC2021\SubArea\database\Subarea.csv"
#subarea_csv_path = r"D:\git\imWEBs\IMWEBsInterface\resources\DemoDatasets\STC2021\SubArea\database\SubareaProcessed.csv"

#BRC
#subarea_data_path = r"D:\imWEBs\BRC\model\BRC\Existing\SubareaData"
#subarea_csv_path = r"D:\git\imWEBs\IMWEBsInterface\resources\DemoDatasets\BRC2021\SubArea\Subarea.csv"
#subarea_csv_path = r"D:\git\imWEBs\IMWEBsInterface\resources\DemoDatasets\BRC2021\SubArea\SubareaProcessed.csv"

#existing, wetland loss, wetland restoration
model_path = r"D:\imWEBs\BRC\model\Existing\Existing"
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_NoBuffer/output"

#wetland enhancement 5m
model_path = r"D:\imWEBs\BRC\model\Enhancement_5m\Enhancement_5m"
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_5m/output"

#wetland enhancement 15m
model_path = r"D:\imWEBs\BRC\model\Enhancement_15m\Enhancement_15m"
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_15m/output"

#wetland enhancement 30m
model_path = r"D:\imWEBs\BRC\model\Enhancement_30m\Enhancement_30m"
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_30m/output"

#don't change from here
#---------------------------------------------------------------------
subarea_data_path = os.path.join(model_path, "SubareaData")
bmp_db3_path = os.path.join(model_path, "bmp.db3")

subarea_csv = os.path.join(output_path, 'subarea.csv')
subarea_csv_processed = os.path.join(output_path, 'subarea_processed.csv')
cell_subarea_csv = os.path.join(output_path, "CellSubarea.csv")
cell_subarea_csv_processed = os.path.join(output_path, "CellSubarea_processed.csv")
subarea_landuse_csv = os.path.join(output_path, 'SubareaLanduseType.csv')
subarea_soil_csv = os.path.join(output_path, 'SubareaSoilType.csv')

#col names
cell_subarea_cell_index_col = "CellIndex"
cell_subarea_cell_subarea_id_col = "SubAreaId"
subarea_id_col = "Id"

In [45]:
# add CellIndex, SubAreaId to export file CellSubarea.csv
print("Add cell index to CellSubarea.csv")
cell_subarea_df = pd.read_csv(cell_subarea_csv, names=[cell_subarea_cell_subarea_id_col])
cell_subarea_df[cell_subarea_cell_index_col] = cell_subarea_df.index
cell_subarea_df[cell_subarea_cell_subarea_id_col] = cell_subarea_df[cell_subarea_cell_subarea_id_col].fillna(-32768)
cell_subarea_df.to_csv(cell_subarea_csv_processed, index=False,columns=[cell_subarea_cell_index_col, cell_subarea_cell_subarea_id_col])

Add cell index to CellSubarea.csv


In [46]:
#read all cell-parameters from text files and calculate the average for each subarea
print("Read cell-level parameters")
for subarea_data_file in os.listdir(subarea_data_path):
    if ".txt" not in subarea_data_file:
        continue

    print(subarea_data_file)

    df = pd.read_csv(os.path.join(subarea_data_path, subarea_data_file), skip_blank_lines=True, names = [getColumnFromFile(subarea_data_file)], delim_whitespace = True)
    df.index.name = cell_subarea_cell_index_col
    
    cell_subarea_df = cell_subarea_df.merge(df, on = cell_subarea_cell_index_col, how = "left")

#set -32768 to null
cell_subarea_df = cell_subarea_df.replace({-32768: np.nan})

#get subarea average
average_df = pd.pivot_table(cell_subarea_df, values=list(file_column_dic.values()), index=cell_subarea_cell_subarea_id_col, aggfunc=np.average)
average_df = average_df.replace({np.nan: 0.0})
average_df.index.name = subarea_id_col

Read cell-level parameters
flow_acc.txt
IUH002Yr_delta_t.txt
IUH002Yr_t0s.txt
IUH010Yr_delta_t.txt
IUH010Yr_t0s.txt
IUH100Yr_delta_t.txt
IUH100Yr_t0s.txt
moist_in.txt
usle_p.txt
wetland.txt


In [47]:
print("add average summary to subarea table")

#merge with id column
subarea_csv_df = pd.read_csv(subarea_csv)
subarea_csv_df = subarea_csv_df.merge(average_df, on = subarea_id_col, how = "left")

#add TopographyWeight and LateralWidth
subarea_csv_df["TopographyWeight"] = 1
subarea_csv_df["LateralWidth"] = np.sqrt(subarea_csv_df["Area"] * 10000)/10

#add value for missing subarea in CellSubarea.csv
subarea_csv_df = subarea_csv_df.fillna(method="ffill")

#save to file
subarea_csv_df.to_csv(subarea_csv_processed, index= False)

add average summary to subarea table


In [48]:
#save the csv to bmp database
print("save to bmp")
conn = sqlite3.connect(bmp_db3_path) 

#subarea
subarea_csv_df.to_sql('SubArea', conn, if_exists='replace', index = False)

#CellSubArea
cell_subarea_df = pd.read_csv(cell_subarea_csv_processed)
cell_subarea_df.to_sql('CellSubarea', conn, if_exists='replace', index = False)

#SubareaLanduseType
subarea_landuse_csv_df = pd.read_csv(subarea_landuse_csv)
subarea_landuse_csv_df.to_sql('SubareaLanduseType', conn, if_exists='replace', index = False)

#SubareaSoilType
subarea_soil_csv_df = pd.read_csv(subarea_soil_csv)
subarea_soil_csv_df.to_sql('SubareaSoilType', conn, if_exists='replace', index = False)

save to bmp
